# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the wine review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import json
import re
import string
import math
from collections import Counter
from IPython.display import display, HTML

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 5

## 1. Load the data <a name="load"></a>

In [ ]:
# Load the full dataset
with open("/app/data/wine-reviews/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)

In [ ]:
wine_data[10]

In [ ]:
# Filter the dataset
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

In [ ]:
# Count the recipes
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")

In [ ]:
example = filtered_data[25]
print(example)

## 2. Tokenize the data <a name="tokenize"></a>

In [ ]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [ ]:
# Display an example of a recipe
example_data = text_data[25]
example_data

In [ ]:
# A DataLoader over a plain list of strings yields shuffled batches
# (lists) of BATCH_SIZE strings
text_ds = DataLoader(text_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Create a vectorisation layer: a small plain-Python replacement for
# Keras' TextVectorization(standardize="lower", output_mode="int")
class TextVectorizer:
    def __init__(self, max_tokens, output_sequence_length):
        self.max_tokens = max_tokens
        self.output_sequence_length = output_sequence_length

    def adapt(self, text_batches):
        # Count every word, then keep the most frequent ones.
        # Index 0 is reserved for padding and 1 for unknown words.
        counts = Counter(
            word for batch in text_batches for text in batch for word in text.lower().split()
        )
        most_common = [word for word, _ in counts.most_common(self.max_tokens - 2)]
        self.vocab = ["", "[UNK]"] + most_common
        self.word_to_index = {word: i for i, word in enumerate(self.vocab)}

    def get_vocabulary(self):
        return self.vocab

    def encode(self, text):
        # words -> ids, then cut / pad with 0 to a fixed length
        ids = [self.word_to_index.get(word, 1) for word in text.lower().split()]
        ids = ids[: self.output_sequence_length]
        return ids + [0] * (self.output_sequence_length - len(ids))

    def __call__(self, texts):
        # one string -> 1D tensor, a list of strings -> 2D tensor (batch, length)
        if isinstance(texts, str):
            return torch.tensor(self.encode(texts))
        return torch.tensor([self.encode(text) for text in texts])


vectorize_layer = TextVectorizer(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_LEN + 1,
)

In [ ]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [ ]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

In [ ]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

## 3. Create the Training Set <a name="create"></a>

In [ ]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(texts):
    tokenized_sentences = vectorize_layer(texts)  # (batch, MAX_LEN + 1)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


# Same loader as text_ds, but collate_fn turns every batch of strings
# into (x, y) token tensors - the equivalent of tf.data's .map()
train_ds = DataLoader(
    text_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=prepare_inputs
)

In [ ]:
example_input_output = next(iter(train_ds))

In [ ]:
# Example Input
example_input_output[0][0]

In [ ]:
# Example Output (shifted by one token)
example_input_output[1][0]

## 5. Create the causal attention mask function <a name="causal"></a>

In [ ]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype, device=None):
    i = torch.arange(n_dest, device=device)[:, None]
    j = torch.arange(n_src, device=device)
    m = i >= j - n_src + n_dest  # True where a token may attend (itself or earlier)
    mask = m.to(dtype)
    # same mask for every sequence in the batch: (batch_size, n_dest, n_src)
    return mask.unsqueeze(0).expand(batch_size, -1, -1)


causal_attention_mask(1, 10, 10, dtype=torch.int32)[0].T

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [ ]:
class MultiHeadAttention(nn.Module):
    """Self-attention written out explicitly (like Keras layers.MultiHeadAttention
    with key_dim per head), so that we can return the attention scores."""

    def __init__(self, num_heads, key_dim, embed_dim):
        super().__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.query = nn.Linear(embed_dim, num_heads * key_dim)
        self.key = nn.Linear(embed_dim, num_heads * key_dim)
        self.value = nn.Linear(embed_dim, num_heads * key_dim)
        self.output = nn.Linear(num_heads * key_dim, embed_dim)

    def forward(self, x, attention_mask):
        batch_size, seq_len, _ = x.shape

        def split_heads(t):  # (B, L, H * K) -> (B, H, L, K)
            return t.view(batch_size, seq_len, self.num_heads, self.key_dim).transpose(1, 2)

        q = split_heads(self.query(x))
        k = split_heads(self.key(x))
        v = split_heads(self.value(x))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.key_dim)  # (B, H, L, L)
        # hide future tokens: their score becomes -inf, i.e. weight 0 after softmax
        scores = scores.masked_fill(~attention_mask[:, None], float("-inf"))
        attention_scores = scores.softmax(dim=-1)

        out = attention_scores @ v  # (B, H, L, K)
        out = out.transpose(1, 2).reshape(batch_size, seq_len, -1)  # (B, L, H * K)
        return self.output(out), attention_scores


class TransformerBlock(nn.Module):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, key_dim, embed_dim)
        self.dropout_1 = nn.Dropout(dropout_rate)
        self.ln_1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.ffn_1 = nn.Linear(embed_dim, ff_dim)
        self.ffn_2 = nn.Linear(ff_dim, embed_dim)
        self.dropout_2 = nn.Dropout(dropout_rate)
        self.ln_2 = nn.LayerNorm(embed_dim, eps=1e-6)

    def forward(self, inputs):
        batch_size, seq_len, _ = inputs.shape
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, torch.bool, device=inputs.device
        )
        attention_output, attention_scores = self.attn(
            inputs, attention_mask=causal_mask
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = torch.relu(self.ffn_1(out1))
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [ ]:
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        maxlen = x.shape[-1]
        positions = torch.arange(0, maxlen, device=x.device)
        positions = self.pos_emb(positions)  # (maxlen, embed_dim)
        x = self.token_emb(x)  # (batch, maxlen, embed_dim)
        return x + positions

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [ ]:
class GPT(nn.Module):
    def __init__(self, max_len, vocab_size, embed_dim, num_heads, key_dim, ff_dim):
        super().__init__()
        self.embedding = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)
        self.transformer = TransformerBlock(num_heads, key_dim, embed_dim, ff_dim)
        # No softmax here: the model outputs logits and nn.CrossEntropyLoss
        # applies the softmax internally
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, inputs):
        x = self.embedding(inputs)
        x, attention_scores = self.transformer(x)
        return self.output(x), attention_scores


gpt = GPT(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM, N_HEADS, KEY_DIM, FEED_FORWARD_DIM).to(device)
optimizer = torch.optim.Adam(gpt.parameters(), lr=1e-3)  # Keras "adam" defaults
# Padding (id 0) is NOT ignored, as in the Keras version: the model learns
# to predict 0 after the text ends, which is what stops generation below
loss_fn = nn.CrossEntropyLoss()

In [ ]:
print(gpt)
print(f"Total params: {sum(p.numel() for p in gpt.parameters()):,}")

In [ ]:
if LOAD_MODEL:
    gpt.load_state_dict(torch.load("./models/gpt.pt", map_location=device))

## 9. Train the Transformer <a name="train"></a>

In [ ]:
# Create a TextGenerator (called after every epoch, like a Keras callback)
class TextGenerator:
    def __init__(self, model, index_to_word, top_k=10):
        self.model = model
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs.astype(np.float64) ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    @torch.no_grad()
    def generate(self, start_prompt, max_tokens, temperature):
        self.model.eval()  # disable dropout
        device = next(self.model.parameters()).device
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = torch.tensor([start_tokens], device=device)
            logits, att = self.model(x)
            y = logits.softmax(dim=-1).cpu().numpy()  # logits -> probabilities
            att = att.cpu().numpy()
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],  # (heads, tokens) for the last token
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

In [ ]:
# Logging and checkpoint saving happen inside the training loop below
writer = SummaryWriter(log_dir="./logs")  # TensorBoard logs

# Tokenize starting prompt
text_generator = TextGenerator(gpt, vocab)

In [ ]:
for epoch in range(EPOCHS):
    gpt.train()
    total_loss = 0.0
    for x, y in train_ds:
        x, y = x.to(device), y.to(device)

        logits, _ = gpt(x)  # (batch, seq_len, vocab)
        # CrossEntropyLoss wants (N, classes): flatten batch and sequence dims
        loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(train_ds)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {epoch_loss:.4f}")
    writer.add_scalar("loss", epoch_loss, epoch)
    torch.save(gpt.state_dict(), "./checkpoint/checkpoint.pt")

    text_generator.generate("wine review", max_tokens=80, temperature=1.0)

writer.close()

In [ ]:
# Save the final model
torch.save(gpt.state_dict(), "./models/gpt.pt")

# 3. Generate text using the Transformer

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [ ]:
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)

In [ ]:
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)

In [ ]:
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)